# Road damage detection with YOLO26m

Attach **DataSprint 2.0: Intelligent Road Damage Detection**, select **GPU T4 x2**, and enable Internet to fetch the starter kit dependencies and the official COCO checkpoint. Select **Save Version → Save & Run All** to run on Kaggle in the background.

Fine-tune COCO-pretrained YOLO26m at 640 px for up to 65 epochs on two available GPUs, with early stopping after 15 epochs without improvement. The notebook compares 640 and 768 px inference on the same 2,150 validation images using the official scorer. The reference validation score is **42.34/100**.

The checkpoint, validation scores, and checked prediction CSV are saved under **/kaggle/working/yolo26m/**.

## 1. Find the competition files

In [ ]:
import csv
import json
import math
import random
import re
import shutil
import subprocess
import sys
import tempfile
from collections import defaultdict
from pathlib import Path

EPOCHS = 65
PATIENCE = 15
IMAGE_SIZE = 640
TRAIN_BATCH_SIZE = 16
PREDICT_BATCH_SIZE = 8
PREDICT_SIZES = (640, 768)
DEGRADED_TRAIN_FRACTION = 0.10
SEED = 42
INITIAL_WEIGHTS = "yolo26m.pt"
CLASSES = ["D00", "D10", "D20", "D40"]
COORDINATES = ["x_min", "y_min", "x_max", "y_max"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/yolo26m")
COMPETITION = "datasprint-2-road-damage-detection"
required_files = (
    "requirements.txt",
    "train_labels.csv",
    "val_labels.csv",
    "val_image_ids.csv",
    "test_image_ids.csv",
    "score.py",
    "make_kaggle_submission.py",
    "check_submission.py",
)


def is_competition_kit(folder):
    return all((folder / name).is_file() for name in required_files) and all(
        (folder / f"{split}_images").is_dir() for split in ("train", "val", "test")
    )


candidates = [INPUT_DIR / "competitions" / COMPETITION, INPUT_DIR / COMPETITION]
KIT = next((folder for folder in candidates if is_competition_kit(folder)), None)
if KIT is None:
    matches = [
        path.parent
        for path in INPUT_DIR.rglob("train_labels.csv")
        if is_competition_kit(path.parent)
    ]
    if len(matches) != 1:
        raise RuntimeError(
            "Attach one copy of the full DataSprint 2.0 kit under Input → Add Input."
        )
    KIT = matches[0]

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
WORK_DIR = Path(tempfile.mkdtemp(prefix="road_damage_"))
print(f"Competition files: {KIT}")

## 2. Install the starter kit dependencies

Packages and the initial COCO checkpoint may need to be fetched on first use. If Internet is unavailable, use a session with the package dependencies and approved checkpoint already available, and set INITIAL_WEIGHTS to the checkpoint path.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(KIT / "requirements.txt")],
    check=True,
)

## 3. Prepare the YOLO dataset

The original images stay in the competition input. This cell creates links and converts pixel boxes to YOLO labels. Photos with no boxes remain in the dataset. The notebook also makes a degraded copy of 10% of the training photos using dimming, blur, noise, or JPEG compression. Each copy keeps the original boxes. Validation and test photos are read as supplied.

In [ ]:
import numpy as np
import torch
import yaml
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU under Session options → Accelerator and rerun.")
gpu_count = torch.cuda.device_count()
training_devices = [0, 1] if gpu_count >= 2 else 0
print(f"Training device(s): {training_devices} ({torch.cuda.get_device_name(0)})")


def index_images(split):
    paths = [
        path
        for path in sorted((KIT / f"{split}_images").iterdir())
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]
    images = {path.stem: path for path in paths}
    if not images or len(images) != len(paths):
        raise ValueError(f"Expected nonempty, unique image filenames in {split}_images.")
    return images


def prepare_split(split, images, dataset_dir):
    annotations = defaultdict(list)
    labels_csv = KIT / f"{split}_labels.csv"
    with labels_csv.open(newline="", encoding="utf-8-sig") as handle:
        for line_number, row in enumerate(csv.DictReader(handle), start=2):
            image_id = Path(row["image_id"].strip()).stem
            class_id = int(row["class_id"])
            if image_id not in images or not 0 <= class_id < len(CLASSES):
                raise ValueError(f"Unknown image or class in {labels_csv.name}:{line_number}")
            coords = tuple(float(row[key]) for key in COORDINATES)
            annotations[image_id].append((class_id, coords))

    image_dir = dataset_dir / "images" / split
    label_dir = dataset_dir / "labels" / split
    image_dir.mkdir(parents=True)
    label_dir.mkdir(parents=True)

    clipped = 0
    discarded = 0
    used = 0
    for image_id, path in tqdm(images.items(), desc=f"Preparing {split}", unit="image"):
        (image_dir / path.name).symlink_to(path)
        lines = []
        if image_id in annotations:
            with Image.open(path) as image:
                width, height = image.size

            for class_id, coords in annotations[image_id]:
                if not all(math.isfinite(value) for value in coords):
                    discarded += 1
                    continue
                x1, y1, x2, y2 = coords
                bounds = (
                    min(max(x1, 0), width),
                    min(max(y1, 0), height),
                    min(max(x2, 0), width),
                    min(max(y2, 0), height),
                )
                x1, y1, x2, y2 = bounds
                if x2 <= x1 or y2 <= y1:
                    discarded += 1
                    continue

                clipped += bounds != coords
                values = (
                    (x1 + x2) / (2 * width),
                    (y1 + y2) / (2 * height),
                    (x2 - x1) / width,
                    (y2 - y1) / height,
                )
                lines.append(f"{class_id} " + " ".join(str(value) for value in values))

        used += len(lines)
        (label_dir / f"{image_id}.txt").write_text("\n".join(lines), encoding="utf-8")

    if used == 0:
        raise ValueError(f"No valid boxes found in {labels_csv.name}.")
    print(f"{split}: {len(images):,} images, {used:,} boxes, {clipped} clipped, {discarded} discarded")


def add_degraded_training_images(images, dataset_dir):
    rng = random.Random(SEED)
    noise_rng = np.random.default_rng(SEED)
    selected = rng.sample(list(images.items()), int(len(images) * DEGRADED_TRAIN_FRACTION))
    train_images_dir = dataset_dir / "images" / "train"
    train_labels_dir = dataset_dir / "labels" / "train"
    effects = ["dim", "blur", "noise", "jpeg"]
    rng.shuffle(effects)

    for index, (image_id, path) in enumerate(tqdm(selected, desc="Adding degraded training photos")):
        effect = effects[index % len(effects)]
        quality = 90
        with Image.open(path) as source:
            image = source.convert("RGB")
        if effect == "dim":
            image = ImageEnhance.Brightness(image).enhance(rng.uniform(0.50, 0.78))
        elif effect == "blur":
            image = image.filter(ImageFilter.GaussianBlur(rng.uniform(0.65, 1.50)))
        elif effect == "noise":
            pixels = np.asarray(image, dtype=np.float32)
            noise = noise_rng.normal(0, rng.uniform(5, 14), size=pixels.shape)
            image = Image.fromarray(np.clip(pixels + noise, 0, 255).astype(np.uint8))
        else:
            quality = rng.randint(22, 42)

        degraded_id = f"degraded_{image_id}"
        image.save(train_images_dir / f"{degraded_id}.jpg", quality=quality)
        shutil.copyfile(
            train_labels_dir / f"{image_id}.txt",
            train_labels_dir / f"{degraded_id}.txt",
        )

    print(f"Added {len(selected):,} degraded copies to the training split.")


def prediction_inputs(split, images):
    with (KIT / f"{split}_image_ids.csv").open(newline="", encoding="utf-8-sig") as handle:
        ids = [row["image_id"].strip() for row in csv.DictReader(handle)]
    if not ids or not all(ids) or len(set(ids)) != len(ids):
        raise ValueError(f"Missing or duplicate IDs in {split}_image_ids.csv.")
    missing = [image_id for image_id in ids if Path(image_id).stem not in images]
    if missing:
        raise FileNotFoundError(f"{split}: {len(missing)} image files missing, including {missing[0]}")
    return [(image_id, images[Path(image_id).stem]) for image_id in ids]


train_images = index_images("train")
val_images = index_images("val")
test_images = index_images("test")
val_inputs = prediction_inputs("val", val_images)
test_inputs = prediction_inputs("test", test_images)
print(f"Prediction lists: {len(val_inputs):,} validation, {len(test_inputs):,} test images")

dataset_dir = Path(tempfile.mkdtemp(prefix="dataset_", dir=WORK_DIR))
prepare_split("train", train_images, dataset_dir)
add_degraded_training_images(train_images, dataset_dir)
prepare_split("val", val_images, dataset_dir)

dataset_yaml = dataset_dir / "data.yaml"
dataset_yaml.write_text(
    yaml.safe_dump(
        {
            "path": str(dataset_dir),
            "train": "images/train",
            "val": "images/val",
            "names": CLASSES,
        }
    ),
    encoding="utf-8",
)

## 4. Train YOLO26m on the available T4 GPUs

In [ ]:
training_dir = Path(tempfile.mkdtemp(prefix="training_", dir=WORK_DIR))
training_model = YOLO(INITIAL_WEIGHTS)
training_batch = TRAIN_BATCH_SIZE if gpu_count >= 2 else TRAIN_BATCH_SIZE // 2
training_model.train(
    data=str(dataset_yaml),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMAGE_SIZE,
    batch=training_batch,
    device=training_devices,
    workers=2,
    seed=SEED,
    project=str(training_dir),
    name="yolo26m",
)

run_dir = training_dir / "yolo26m"
best_weights = OUTPUT_DIR / "best.pt"
shutil.copy2(run_dir / "weights" / "best.pt", best_weights)
shutil.copy2(run_dir / "results.csv", OUTPUT_DIR / "training_metrics.csv")
del training_model
print(f"Best checkpoint: {best_weights}")

## 5. Compare 640 px and 768 px on the official validation set

The notebook retains predictions down to 0.001, with up to 100 boxes per image. It chooses the inference size with the higher official validation accuracy score. If the rounded scores tie, it keeps 640 px.

In [ ]:
def export_predictions(model, inputs, destination, image_size):
    fieldnames = ["image_id", "class_id", "score", *COORDINATES]
    box_count = 0
    offsets = range(0, len(inputs), PREDICT_BATCH_SIZE)
    with destination.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(fieldnames)
        for offset in tqdm(offsets, desc=destination.stem, unit="batch"):
            batch = inputs[offset : offset + PREDICT_BATCH_SIZE]
            results = model.predict(
                source=[str(path) for _, path in batch],
                imgsz=image_size,
                conf=0.001,
                iou=0.7,
                max_det=100,
                device=0,
                verbose=False,
            )

            for (image_id, path), result in zip(batch, results, strict=True):
                if Path(result.path).name != path.name:
                    raise ValueError(f"Prediction returned {result.path} for {path.name}.")
                if result.boxes is None:
                    continue
                for x1, y1, x2, y2, score, class_id in result.boxes.data.cpu().tolist():
                    if x2 > x1 and y2 > y1:
                        writer.writerow([image_id, int(class_id), score, x1, y1, x2, y2])
                        box_count += 1

    print(f"{destination.name}: {len(inputs):,} images, {box_count:,} predicted boxes")




def run_kit_script(script, *args):
    result = subprocess.run(
        [sys.executable, str(KIT / script), *(str(arg) for arg in args)],
        cwd=WORK_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf-8",
        errors="replace",
    )
    print(result.stdout, end="")
    result.check_returncode()
    return result.stdout


def accuracy_score(report):
    match = re.search(
        r"(?im)^\s*accuracy score\s*:\s*(\d+(?:\.\d+)?)\s*/\s*100\b",
        report,
    )
    if match is None:
        raise ValueError("The official scorer did not print an accuracy score out of 100.")
    return float(match.group(1))


detector = YOLO(str(best_weights))
validation = []
for image_size in PREDICT_SIZES:
    print(f"\nOfficial validation at {image_size} px")
    val_predictions = WORK_DIR / f"val_pred_{image_size}.csv"
    export_predictions(detector, val_inputs, val_predictions, image_size)
    report = run_kit_script(
        "score.py",
        "--pred", val_predictions,
        "--gt", KIT / "val_labels.csv",
        "--image-ids", KIT / "val_image_ids.csv",
    )
    validation.append({"image_size": image_size, "accuracy_score": accuracy_score(report)})

best_validation = max(validation, key=lambda result: result["accuracy_score"])
selected_size = best_validation["image_size"]
summary = {
    "checkpoint": best_weights.name,
    "validation_scores": validation,
    "selected_inference": best_validation | {"confidence_floor": 0.001, "iou": 0.7, "max_det": 100},
}
(OUTPUT_DIR / "validation_scores.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8"
)
print(f"Selected {selected_size} px with validation score {best_validation['accuracy_score']:.2f}/100.")

## 6. Generate and check the Kaggle submission

The notebook uses the inference size selected on validation. The official converter writes one row per test image and uses the word none for images without detections. Open the Output pane after the checker passes and upload submission.csv on the competition's submission page.

In [ ]:
test_predictions = WORK_DIR / "test_pred.csv"
test_ids = KIT / "test_image_ids.csv"
candidate = WORK_DIR / "submission.csv"

export_predictions(detector, test_inputs, test_predictions, selected_size)
run_kit_script(
    "make_kaggle_submission.py",
    "--pred", test_predictions,
    "--image-ids", test_ids,
    "--out", candidate,
)
run_kit_script("check_submission.py", "--pred", candidate, "--image-ids", test_ids)

submission = OUTPUT_DIR / "submission.csv"
shutil.copy2(candidate, submission)
print(f"Checked submission for {len(test_inputs):,} images: {submission}")